# Replica staleness through Pgpool-II

**Question.** When a write goes through Pgpool on one connection, can a read on a *different*
connection fail to see it?

This is the mechanism behind the hypothesis that `local-k8s` `.memory/active-issues.md` **#15**
was once labelled with — "PG-HA read-after-write signature" — a label that was asserted and
never established. What the Polaris logs actually carry is a `NullPointerException` from
`IcebergExceptionMapper`, and the trigger looks like a *fresh parent entity*, not load. This
notebook settles the database half of it, on its own, without Polaris.

**Why it can happen at all**, from `local-k8s/postgresql/values.yaml` and the
`postgresql-ha` 16.3.2 defaults:

| link | setting | value |
|---|---|---|
| replication is asynchronous | `postgresql.syncReplication` (never set) | `false` |
| reads are load-balanced onto standbys | `pgpool.useLoadBalancing` (never set) | `true` |
| the read backend is fixed **per session**, not per statement | `statement_level_load_balance` | `off` |
| write-pinning protects only **within** one session | `disableLoadBalancingOnWrite` | `always` |
| **replication lag is never checked** | `delay_threshold` — set nowhere in chart, templates or values | pgpool default `0` |

`disable_load_balance_on_write = always` pins *a session* to the primary after that session
writes. Polaris writes on one Agroal pooled connection and resolves on another — a different
pgpool session, no write of its own, load-balanced, and with no `delay_threshold` it can be sent
to a standby that is arbitrarily far behind.

**Method.** Pause WAL *replay* on every standby so staleness is guaranteed rather than hoped for,
write one row through Pgpool, then read from N brand-new pooled sessions. Each read reports
`pg_is_in_recovery()` — which tells us *which node served it* — alongside the row count.

## ⚠ Read before running

- **Local only.** This pauses replication. `require_not_prod()` blocks `prod`; the extra
  assertion below also blocks `dev`, because that is a shared company cluster.
- **Keep the paused window to minutes.** Replay pauses; WAL keeps arriving. The pre-flight cell
  prints which retention regime you are in and what the risk is.
- **If anything goes wrong**, run the PANIC cell. It is idempotent and works standalone after a
  kernel restart.
- Requires `kubectl` on this machine with the context set to `orbstack`.

## 1 — Setup and guards

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from polaris_test_utils import *  # noqa: F401,F403

init_env("local")
require_not_prod("pause WAL replay on the PostgreSQL standbys")

# require_not_prod only blocks `prod`. `dev` is a SHARED company cluster and
# pausing replay there would degrade other people's reads, so this notebook is
# local-only with no override.
assert ENV == "local", (
    f"Refusing to run against env={ENV!r}. This notebook pauses WAL replay and "
    "is local-only by design."
)

print(f"env      = {ENV}")
print(f"postgres = {PG_HOST}:{PG_PORT}/{PG_DB} as {PG_USER}")

In [ ]:
import json
import subprocess
import time

NAMESPACE = "datahub-hynix"
STS = "benchmarks-postgresql-postgresql-ha-postgresql"
PGPOOL_SVC = "benchmarks-postgresql-postgresql-ha-pgpool"
CONTAINER = "postgresql"
PW_FILE = "/opt/bitnami/postgresql/secrets/postgres-password"
REPLICAS = 3
PROBE_TABLE = "rw_probe"
N_SESSIONS = 12
EXPECT_CTX = "orbstack"

ctx = subprocess.run(
    ["kubectl", "config", "current-context"], capture_output=True, text=True
).stdout.strip()
assert ctx == EXPECT_CTX, f"kubectl context is {ctx!r}, expected {EXPECT_CTX!r}"
print(f"kubectl context = {ctx}")

# PG_HOST must be the Pgpool LoadBalancer, not one pod. The whole experiment is
# about how Pgpool routes; pointed at a pod, every read lands on that pod and
# the result means nothing.
print(f"PG_HOST         = {PG_HOST}")
print(f"                  ^ must be the {PGPOOL_SVC} LoadBalancer, NOT a single pod")

## 2 — Helpers

In [ ]:
import psycopg2


def kexec(ordinal, sql, db=None):
    """Run SQL as superuser directly on one Postgres pod, bypassing Pgpool.

    The password is read from the file the chart already mounts inside the pod,
    so no credential appears anywhere in this notebook.
    """
    db = db or PG_DB
    inner = f'PGPASSWORD=$(cat {PW_FILE}) psql -U postgres -d {db} -tAc "$0"'
    r = subprocess.run(
        [
            "kubectl", "-n", NAMESPACE, "exec", f"{STS}-{ordinal}",
            "-c", CONTAINER, "--", "sh", "-c", inner, sql,
        ],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"pg-{ordinal}: {r.stderr.strip()}")
    return r.stdout.strip()


def pool_sql(sql, fetch=True):
    """Run SQL through Pgpool on a BRAND NEW connection, then close it.

    Deliberately unpooled, and a deliberate exception to this repo's
    close-the-pool convention: there is no pool here to close. Because
    `statement_level_load_balance = off`, Pgpool chooses the read backend once
    per session and keeps it. Reusing a connection would therefore ask the same
    backend every time and measure nothing.

    One call == one session == one independent routing decision, which is
    exactly the shape Polaris produces: an Agroal pool of 10-300 connections,
    each of them its own Pgpool session.
    """
    conn = psycopg2.connect(**PG_CONFIG)
    try:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall() if fetch else None
    finally:
        conn.close()


print("helpers ready")

## 🚨 PANIC / RECOVERY

**Run this cell on its own if anything goes wrong** — including after a kernel restart, in which
case run cells 1 and 2 first. It is idempotent: it resumes WAL replay on every pod that is
currently in recovery and leaves the primary alone.

In [ ]:
def resume_all(verbose=True):
    """Resume WAL replay on every standby. Safe to call any number of times."""
    out = []
    for i in range(REPLICAS):
        try:
            if kexec(i, "SELECT pg_is_in_recovery();") != "t":
                out.append((i, "primary - nothing to do"))
                continue
            kexec(i, "SELECT pg_wal_replay_resume();")
            out.append((i, kexec(i, "SELECT pg_get_wal_replay_pause_state();")))
        except Exception as e:  # noqa: BLE001 - recovery path must not raise
            out.append((i, f"ERROR: {e}"))
    if verbose:
        for i, s in out:
            print(f"pg-{i}: {s}")
    return out


# resume_all()   # <- uncomment and run THIS CELL ONLY if a run was interrupted

## 3 — Who is the primary?

Derived, never assumed. On 2026-09-17 the primary was **pg-1**, not pg-0 — a hand-written script
that hardcoded `for i in 1 2` tried to pause the primary and left a standby running, which
silently invalidated the run.

In [ ]:
import pandas as pd

rows = []
for i in range(REPLICAS):
    in_recovery = kexec(i, "SELECT pg_is_in_recovery();") == "t"
    rows.append(
        {
            "ordinal": i,
            "pod": f"{STS}-{i}",
            "role": "standby" if in_recovery else "primary",
            "addr": kexec(i, "SELECT inet_server_addr();"),
        }
    )

roles = pd.DataFrame(rows)
display(roles)

primaries = roles.loc[roles.role == "primary", "ordinal"].tolist()
STANDBYS = roles.loc[roles.role == "standby", "ordinal"].tolist()
assert len(primaries) == 1, f"expected exactly one primary, found {primaries}"
assert STANDBYS, "no standbys found - nothing to make stale"
PRIMARY = primaries[0]

print(f"primary = pg-{PRIMARY}   standbys = {STANDBYS}")

## 4 — Pre-flight: which WAL-retention risk are you taking?

Pausing replay stops *replay*, not *receipt*. WAL keeps arriving and accumulating, and the
consequence depends on whether replication slots are in use.

In [ ]:
slots = int(kexec(PRIMARY, "SELECT count(*) FROM pg_replication_slots;"))
wal_keep = kexec(PRIMARY, "SHOW wal_keep_size;")

print(f"replication slots on primary : {slots}")
print(f"wal_keep_size                : {wal_keep}\n")

if slots > 0:
    print("  SLOTS PRESENT -> the primary retains WAL for a paused standby WITHOUT BOUND.")
    print("  Watch the PVC below. Keep the paused window to minutes.")
else:
    print(f"  NO SLOTS -> retention is capped at {wal_keep}. If a paused standby falls further")
    print("  behind than that it CANNOT catch up on resume and needs a full re-clone")
    print("  (`usePgRewind` is false on this chart, so it is pg_basebackup, not a rewind).")
    print("  Keep the paused window to minutes.")

print("\ndisk on the primary:")
print(
    subprocess.run(
        ["kubectl", "-n", NAMESPACE, "exec", f"{STS}-{PRIMARY}", "-c", CONTAINER,
         "--", "df", "-h", "/bitnami/postgresql"],
        capture_output=True,
        text=True,
    ).stdout
)

## 5 — Plant the probe table, and confirm it replicated

Created **through Pgpool** (a write, so it lands on the primary) in the `polaris` database, as
the `polaris` user — the same database and role the reads will use. A previous hand-run created
the table as `postgres` in the `postgres` database and then read it as `polaris` in `polaris`,
which produced twelve identical `relation "rw_probe" does not exist` errors and no result.

The table is created *before* the pause so that the experiment measures **row visibility**, not
DDL visibility.

In [ ]:
pool_sql(
    f"CREATE TABLE IF NOT EXISTS {PROBE_TABLE} "
    "(id serial PRIMARY KEY, t timestamptz DEFAULT now());",
    fetch=False,
)
time.sleep(2)

seen = {i: kexec(i, f"SELECT to_regclass('{PROBE_TABLE}') IS NOT NULL;") for i in STANDBYS}
print({f"pg-{i}": v for i, v in seen.items()})
assert all(v == "t" for v in seen.values()), (
    f"probe table has not replicated to every standby: {seen}. Do not pause yet."
)
print(f"\n'{PROBE_TABLE}' exists on every standby - safe to pause")

## 6 — Pause replay on **every** standby

`pg_wal_replay_pause()` is a *request*; replay stops at the next recovery pause point. So the
state is checked rather than assumed — `pg_get_wal_replay_pause_state()` distinguishes
`pause requested` from `paused`, which the older boolean `pg_is_wal_replay_paused()` cannot.

**One standby left running serves fresh reads and masks the whole effect**, so this asserts on
all of them.

In [ ]:
for i in STANDBYS:
    kexec(i, "SELECT pg_wal_replay_pause();")
time.sleep(1)

states = {i: kexec(i, "SELECT pg_get_wal_replay_pause_state();") for i in STANDBYS}
print({f"pg-{i}": v for i, v in states.items()})
assert all(v == "paused" for v in states.values()), (
    f"not every standby is paused: {states}. 'pause requested' means replay has not "
    "stopped yet - wait a moment and re-run this cell."
)
print("\nall standbys paused - the clock is running, finish and resume promptly")

## 7 — The experiment

One write through Pgpool, ground truth read straight off the primary, then `N_SESSIONS`
brand-new pooled sessions. `pg_is_in_recovery()` in the same statement tells us which node served
that session, so routing and staleness are measured together.

In [ ]:
before = int(kexec(PRIMARY, f"SELECT count(*) FROM {PROBE_TABLE};"))
pool_sql(f"INSERT INTO {PROBE_TABLE} DEFAULT VALUES;", fetch=False)
truth = int(kexec(PRIMARY, f"SELECT count(*) FROM {PROBE_TABLE};"))
print(f"rows on the primary: {before} -> {truth}   (ground truth = {truth})\n")

observations = []
for n in range(1, N_SESSIONS + 1):
    (in_recovery, n_rows, addr), = pool_sql(
        "SELECT pg_is_in_recovery(), count(*), inet_server_addr()::text "
        f"FROM {PROBE_TABLE};"
    )
    observations.append(
        {
            "session": n,
            "served_by": "standby" if in_recovery else "primary",
            "addr": addr,
            "rows_seen": n_rows,
            "stale": n_rows < truth,
        }
    )

probe = pd.DataFrame(observations)
display(probe)

## 8 — Verdict

In [ ]:
n_standby = int((probe.served_by == "standby").sum())
n_primary = int((probe.served_by == "primary").sum())
n_stale = int(probe.stale.sum())

by_node = (
    probe.groupby(["served_by", "addr"])
    .agg(
        sessions=("session", "count"),
        min_rows=("rows_seen", "min"),
        max_rows=("rows_seen", "max"),
        stale=("stale", "sum"),
    )
    .reset_index()
)
display(by_node)

if n_stale > 0:
    verdict = (
        f"CONFIRMED - {n_stale}/{N_SESSIONS} pooled sessions read a stale row count against a "
        f"ground truth of {truth}; {n_standby}/{N_SESSIONS} sessions were served by a standby. "
        "Pgpool load-balances reads onto replicas with no delay_threshold, so a write on one "
        "Agroal connection is invisible to a read on another. Remedy: "
        "database_redirect_preference_list = 'polaris:primary' in pgpool.configuration - the "
        "'primary' keyword, NOT a node id, which would pin reads to a demoted standby after a "
        "failover."
    )
elif n_standby == 0:
    verdict = (
        f"REFUTED - all {N_SESSIONS} sessions were served by the PRIMARY, so Pgpool is not "
        "load-balancing reads here at all and the read-after-write mechanism cannot explain the "
        "#15 500s. Before concluding, confirm load_balance_mode and the backend weights from the "
        "RUNNING pgpool.conf, not from the values file."
    )
else:
    verdict = (
        f"INCONCLUSIVE - {n_standby}/{N_SESSIONS} sessions hit a standby but none read stale "
        f"(every one saw {truth}). Replay is supposed to be paused; re-check "
        "pg_get_wal_replay_pause_state() on every standby before trusting this."
    )

print(verdict)
print(
    f"\nobserved load-balance ratio: standby {n_standby}/{N_SESSIONS}, "
    f"primary {n_primary}/{N_SESSIONS}"
)
print(
    "that ratio is itself useful - it estimates the fraction of Polaris's pooled "
    "connections reading from a standby at any moment"
)

## 9 — Resume, verify, tear down

In [ ]:
resume_all()
time.sleep(3)

print("\nreplication after resume (client | state | replay_lag):")
print(
    kexec(
        PRIMARY,
        "SELECT client_addr || ' | ' || state || ' | ' || "
        "coalesce(replay_lag::text, '0') FROM pg_stat_replication;",
    )
)

pool_sql(f"DROP TABLE IF EXISTS {PROBE_TABLE};", fetch=False)
print(f"\ndropped {PROBE_TABLE}")

In [ ]:
import gc

try:
    del probe, by_node, roles
except NameError:
    pass
gc.collect()

# No cursors, engines or pools are held: pool_sql opens and closes one
# connection per call by design, and kexec speaks to kubectl, not to a driver.
print("teardown complete - nothing left open")

## How to read the result

| outcome | meaning | what to do |
|---|---|---|
| **CONFIRMED** | stale reads through Pgpool are real | set `database_redirect_preference_list = 'polaris:primary'`; then re-run the #15 catalog ladder to see how much of the 500 rate it removes |
| **REFUTED** | every read went to the primary | the mechanism does not exist here; #15's NPEs are Polaris's own resolver, and since Polaris is not to be changed, the remedy is a client-side retry |
| **INCONCLUSIVE** | reads hit standbys but none were stale | the pause did not take; re-check and re-run before drawing anything |

**A CONFIRMED result proves the mechanism, not the diagnosis.** #15's strongest evidence is that
**3 of 3 brand-new catalogs** 500 on the first namespace created in them — a 3-of-3 pattern is not
what probabilistic replication lag produces. So even with staleness confirmed here, the next step
is to measure how much of the real 500 rate it accounts for. Note also that under normal load the
standbys were measured `streaming` at ~zero lag, so the natural staleness window is small.

**The denominator problem is separate and unsolved.** Per `local-k8s` #15, a *successful*
namespace create leaves no individual access-log record, so the 500 rate cannot be computed from
the log pipeline at all — it has to come from the driver recording its own statuses.

## Related

- `local-k8s/postgresql/HANDOFF-toxiproxy-failover-2026-09-17.md` — the primary-unavailable
  harness, deliberately deferred until this question is settled
- `local-k8s/.memory/active-issues.md` #15 (the 500s), #9 (plaintext JDBC password)
- `local-k8s/postgresql/values.yaml` — `pgpool.configuration` is where the remedy goes